# **Adding Observability with LangSmith**

## **What's Covered?**
1. Introduction to LangSmith
    - What is LangSmith?
    - What is Observability and Telemetry Data?
    - Important Terminology
    - What does LangSmith Records?
    - Setting up LangSmith
2. Example Implementation
    - Step 1: Add the required LangSmith environment variables
    - Step 2: Create a Chain
    - Step 3: Invoking the Chain
    - Step 4: Adding the tags and metadata
3. Example Implementation 2

## **Introduction to LangSmith**

### **What is LangSmith?**
Provides observability & evaluation which helps to debug, test and monitor AI systems/workflows. Eg: Identify why your production workflow is taking more time (i.e. latency), measure the cost, token usage, hallucination, etc...

### **What is Observability?**
Ability to understand a system's internal state by analyzing its external outputs, primarily through telemetry data (i.e. logs, metrics and traces)
- **Logs:** Chronological records of events, actions and messages from the system
- **Metrics:** Numerical measurement over time eg: CPU usage, number of requests, latency, etc...
- **Traces:** Records the journey of a single request as it moves across different services in a distributed system, showing timing and flow.

### **Important Terminology**
- **Run:** Each step within a trace is represented by a run. A run is a span representing a single unit of work or operation within your LLM application. This could be anything from a single call to an LLM or chain, to a prompt formatting call, to a runnable lambda invocation.
- **Thread:** A thread is a sequence of traces representing a single conversation. Many LLM applications have a chatbot-like interface in which the user and the LLM application engage in a multi-turn conversation. Each turn in the conversation is represented as its own trace, but these traces are linked together by being part of the same thread.
- **Projects:** A project is a collection of traces. You can think of a project as a container for all the traces that are related to a single application or service. You can have multiple projects, and each project can have multiple traces.
- **Feedback:** Feedback allows you to score an individual run based on certain criteria. Each feedback entry consists of a feedback tag and feedback score, and is bound to a run by a unique run ID. Feedback can be continuous or discrete (categorical), and you can reuse feedback tags across different runs within an organization.
- **Tags:** Tags are collections of strings that can be attached to runs. You can use tags to do the following in the LangSmith UI: Categorize runs for easier search, Filter runs and Group runs together for analysis.

### **What does LangSmith records?**
1. Inputs and Outputs
2. Token Usage
3. Cost
4. Latency
5. Errors
6. Intermediate steps
7. Tags
8. Metadata
9. Feedback

[Click Here](https://docs.langchain.com/langsmith/observability-concepts) to read the official LangSmith documentation.

### **Setting up LangSmith**

You need to setup the following environment variables initially:
1. LANGSMITH_TRACING=true
2. LANGSMITH_ENDPOINT="https://api.smith.langchain.com"
3. LANGSMITH_API_KEY=`"<api-key>"`
4. LANGSMITH_PROJECT=`"<project-name>"`

In [3]:
! pip show langchain-core

Name: langchain-core
Version: 1.5.4
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: C:\Users\krupa\anaconda3\Lib\site-packages
Requires: jsonpatch, langchain-protocol, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: langchain, langchain-classic, langchain-community, langchain-groq, langchain-text-splitters, langgraph, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk


In [4]:
! pip show langsmith

Name: langsmith
Version: 0.10.18
Summary: Client library to connect to the LangSmith Observability and Evaluation Platform.
Home-page: https://smith.langchain.com/
Author: 
Author-email: LangChain <support@langchain.dev>
License: MIT
Location: C:\Users\krupa\anaconda3\Lib\site-packages
Requires: anyio, distro, httpx, orjson, packaging, pydantic, requests, requests-toolbelt, sniffio, typing-extensions, uuid-utils, websockets, xxhash, zstandard
Required-by: langchain-classic, langchain-community, langchain-core


## **Example Implementation**

### **Step 1: Add the required LangSmith environment variables**

In [7]:
import os

# Setup LANGSMITH API Key
#f = open('keys/.langsmith_api_key.txt')
with open("keys/.langsmith_api_key.txt") as f:
    LANGSMITH_API_KEY = f.read()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
os.environ["LANGSMITH_PROJECT"] = "first-langsmith-observability-project"

### **Step 2: Create a Chain**

In [9]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate(
    messages=[
        ("system", """You are a strict JSON generator.
        Analyze the following customer feedback.
        
        Return STRICT JSON with:
        - sentiment (positive, neutral, negative)
        - category (product, delivery, support, pricing, other)
        - issue (short summary)
        
        Only return JSON. No explanation."""),
        
        ("human", """Feedback:
        ```{feedback}```""")
    ]
)

In [10]:
# TODO: UPDATE ".groq_api_key.txt"

from langchain_groq import ChatGroq

with open("keys/.groq_api_key.txt") as f:
    GROQ_API_KEY = f.read()

groq_gpt_chat_model = ChatGroq(
    api_key=GROQ_API_KEY, 
    model="openai/gpt-oss-120b", 
    temperature=1
)

In [11]:
from pydantic import BaseModel
from typing import Literal

class FeedbackOutput(BaseModel):
    sentiment: Literal["positive", "neutral", "negative"]
    category: Literal["product", "delivery", "support", "pricing", "other"]
    issue: str

In [12]:
from langchain_core.output_parsers import PydanticOutputParser

output_parser = PydanticOutputParser(pydantic_object=FeedbackOutput)

In [13]:
chain = template | groq_gpt_chat_model | output_parser

### **Step 3: Invoking the Chain**

**Note: By default, LangSmith is going to track all the runs which are called using .invoke() method.**

<img src="images/project_dashboard.png">

In [15]:
feedback = "The product quality is good but delivery was very late"

chain.invoke({"feedback" : feedback})

FeedbackOutput(sentiment='negative', category='delivery', issue='Late delivery')

In [16]:
feedback = "The product quality was very bad"

chain.invoke({"feedback" : feedback})

FeedbackOutput(sentiment='negative', category='product', issue='poor quality')

### **Step 4: Adding the tags and metadata**

<img src="images/run_openai.png">

In [18]:
# Configuring the Run Name, Tags and Metadata
config_1 = {
    "run_name": "gpt chain",
    "tags": ["groq", "gpt-oss-120b"],
    "metadata": {"inference": "groq", "llm": "gpt-oss-120b"}
}

feedback = "The product quality is good but delivery was very late"

chain.invoke({"feedback" : feedback}, config=config_1)

FeedbackOutput(sentiment='neutral', category='delivery', issue='delivery was very late')

## **Example Implementation 2**

<img src="images/run_groq.png">

In [20]:
# TODO: Setup Another LLM Inference Provider and Track Model Calls